## ML-Specific Patterns

In [47]:
import numpy as np

In [48]:
# Exercise 8.1

def relu(x: np.ndarray) -> np.ndarray:
    """max(x, 0) element-wise"""
    return np.maximum(0, x)

def leaky_relu(x: np.ndarray, alpha: float = 0.01) -> np.ndarray:
    """x if x > 0 else alpha*x"""
    return np.maximum(x, alpha * x)

def sigmoid(x: np.ndarray) -> np.ndarray:
    """1 / (1 + exp(-x))"""
    return np.where(x >= 0, 1/(1 + np.exp(-x)), np.exp(x) / (1 + np.exp(x)))

def tanh(x: np.ndarray) -> np.ndarray:
    """(exp(x) - exp(-x)) / (exp(x) + exp(-x))"""
    # Use np.tanh but also can do 2 * sigmoid(2 * x) - 1
    return np.tanh(x)

def softmax(x: np.ndarray) -> np.ndarray:
    """exp(x) / sum(exp(x)), stable version, works on 2D batches"""
    shifted = x - x.max(axis=-1, keepdims=True)
    exp_z = np.exp(shifted)
    return exp_z / exp_z.sum(axis=-1, keepdims=True)

def gelu(x: np.ndarray) -> np.ndarray:
    """x * Phi(x) where Phi is standard normal CDF"""
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))


In [49]:
x = np.array([-3, -1, 0, 1, 3], dtype=float)
assert np.all(relu(x) >= 0)
assert relu(np.array([0.0]))[0] == 0.0
assert relu(np.array([5.0]))[0] == 5.0

assert np.all((sigmoid(x) > 0) & (sigmoid(x) < 1))
assert np.isclose(sigmoid(np.array([0.0]))[0], 0.5)

assert np.all((tanh(x) > -1) & (tanh(x) < 1))
assert np.isclose(tanh(np.array([0.0]))[0], 0.0)

batch = np.array([[1.0, 2.0, 3.0], [4.0, 1.0, 2.0]])
sm = softmax(batch)
assert sm.shape == (2, 3)
assert np.allclose(sm.sum(axis=1), [1.0, 1.0])
assert np.all(sm > 0)

In [ ]:
# Exercise 8.2

def mse_loss(y_pred: np.ndarray, y_true: np.ndarray) -> float:
    """Mean squared error: mean((y_pred - y_true)^2)"""
    return np.mean((y_pred - y_true) ** 2)

def mae_loss(y_pred: np.ndarray, y_true: np.ndarray) -> float:
    """Mean absolute error: mean(|y_pred - y_true|)"""
    return np.mean(np.abs(y_pred - y_true))

def binary_cross_entropy(y_pred_prob: np.ndarray, y_true: np.ndarray) -> float:
    """
    Binary cross-entropy.
    y_pred_prob: predicted probabilities, shape (n,) values (0, 1)
    y_true: binary labels, shape (n,) values 0 or 1
    """

    eps = 1e-9
    return -np.mean(y_true*np.log(y_pred_prob+eps) + (1-y_true)*np.log(1-y_pred_prob+eps))

def categorical_cross_entropy(y_pred_probs: np.ndarray, y_true_onehot: np.ndarray) -> float:
    """
    Multi-class cross-entropy.
    y_pred_probs: softmax output, shape (n, k)
    y_true_onehot: one-hot labels, shape (n, k)
    """
    
    eps = 1e-9
    per_sample = -np.sum(y_true_onehot * np.log(y_pred_probs + eps), axis=1)
    return np.mean(per_sample)